### Transform Addresses Data
- Write One Record for each customer with 2 sets of address columns, 1 for shipping and 1 for billing address

- Write the transformed data to the Silver Layer

#### Write One Record for each customer with 2 sets of address columns, 1 for shipping and 1 for billing address

In [0]:
%sql
select * from gizmobox.bronze.v_addresses;

In [0]:
%sql

select 
   e.customer_id, 
   e.address_line_1 as billing_address, 
   f.address_line_1 as shipping_address,
   e.city,e.state,
   e.postcode,
   e.`_rescued_data` 
from gizmobox.bronze.v_addresses e
join gizmobox.bronze.v_addresses f on e.customer_id = f.customer_id and e.address_type = 'billing' and f.address_type = 'shipping'

In [0]:
%sql
select * 
from (
    select 
         customer_id,
         address_type,
         address_line_1,
         city,
         state,
         postcode
    from gizmobox.bronze.v_addresses
)
pivot (
    max(address_line_1) as address_line_1,
    max(city) as city,
    max(state) as state,
    max(postcode) as postcode
    for address_type IN ('billing','shipping')
)

#### Write the transformed data to the Silver Layer

In [0]:
%sql
CREATE OR REPLACE TABLE gizmobox.silver.addresses
as 
select * 
from (
    select 
         customer_id,
         address_type,
         address_line_1,
         city,
         state,
         postcode
    from gizmobox.bronze.v_addresses
)
pivot (
    max(address_line_1) as address_line_1,
    max(city) as city,
    max(state) as state,
    max(postcode) as postcode
    for address_type IN ('billing','shipping')
)

In [0]:
%sql
select * from gizmobox.silver.addresses
order by customer_id;

In [0]:
%sql
desc extended gizmobox.silver.addresses